# Tokopedia Scraper Ultimate (GQL Mobile Payload)

In [2]:
%pip install curl-cffi pandas tqdm -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time
import random
import pandas as pd
import re
from curl_cffi import requests
from tqdm.notebook import tqdm

# ==========================================
# KONFIGURASI PENCARIAN
# ==========================================
KEYWORDS = [
    # Makanan & Kebutuhan Sehari-hari
    "Minyak Goreng 2 Liter", "Susu Formula Bayi", "Popok Bayi Celana", "Sabun Cuci Pakaian",
    "Shampoo Anti Ketombe", "Pasta Gigi Whitening", "Deterjen Cair", "Pembersih Lantai",
    
    # Kesehatan & Hewan Peliharaan
    "Snack Kucing", "Makanan Anjing", "Vitamin C 1000mg", "Madu Asli", "Masker Medis",
    "Termometer Digital", "Timbangan Badan Digital", "Obat Tetes Mata",
    
    # Otomotif
    "Helm Full Face", "Jas Hujan Ponco", "Oli Motor Matic", "Wiper Mobil",
    "Sarung Tangan Motor", "Cairan Pembersih Mesin", "Kampas Rem Motor",
    
    # Fashion Wanita & Pakaian
    "Baju Kaos Oversize", "Celana Jeans Wanita", "Hijab Segi Empat", "Mukena Dewasa",
    "Sandal Gunung Pria", "Dompet Kulit Pria", "Kemeja Flanel", "Ikat Pinggang Kulit",
    
    # Perabotan Dapur & Rumah Tangga
    "Panci Set Stainless", "Spatula Silikon", "Wajan Anti Lengket", "Botol Minum 2 Liter",
    "Kasur Busa", "Sprei Aesthetic", "Lampu Led Kamar", "Meja Belajar Lipat",
    "Kursi Kantor Ergonomis", "Karpet Lantai Bulu", "Lemari Pakaian Plastik", "Cermin Dinding",
    
    # Aksesoris Komputer & HP (Baru)
    "Earphone Kabel", "Cooling Pad Laptop", "Mousepad Gaming", "Flashdisk 64gb",
    "Tripod Handphone", "Mic Clip On", "Ring Light", "Kabel Hdmi"
]


TARGET_TOTAL_ROWS = 20000
TARGET_PER_KEYWORD = TARGET_TOTAL_ROWS // len(KEYWORDS)  # ~1800 baris per keyword

# MENGGUNAKAN SORT OPTIONS UNTUK BYPASS PAGINATION (Halaman 2)
SORT_OPTIONS = [
    23, # Paling Sesuai
    9,  # Terlaris
    3,  # Ulasan Terbanyak
    4,  # Terbaru
    8,  # Harga Tertinggi
    5   # Harga Terendah
]

REVIEWS_PER_PRODUCT = 40 # Ambil hingga 40 ulasan per produk
OUTPUT_FILE = "dataset\\raw-2_tokopedia_dataset.csv"

print("Library dimuat. Target Total Baris:", TARGET_TOTAL_ROWS)
print("Target Baris Per Kategori:", TARGET_PER_KEYWORD)

Library dimuat. Target Total Baris: 20000
Target Baris Per Kategori: 392


## Konfigurasi Kredensial
Header dan Cookie diambil langsung dari *payload* milik Anda.

In [4]:
HEADERS = {
    'sec-ch-ua-platform': '"Windows"',
    'sec-ch-ua': '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
    'x-price-center': 'true',
    'bd-device-id': '7647226487314482689',
    'sec-ch-ua-mobile': '?0',
    'accept': '*/*',
    'content-type': 'application/json',
    'bd-web-id': '7647226487314482689',
    'x-version': '7149a47',
    'x-source': 'tokopedia-lite',
    'x-dark-mode': 'false',
    'tkpd-userid': '12787827',
    'x-device': 'mobile',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/148.0.0.0 Safari/537.36',
    'x-tkpd-lite-service': 'phoenix',

}

session = requests.Session(impersonate="chrome110")

Q_SEARCH = "query SearchProductV5Query($searchProductV5Param: String!) { searchProductV5(params: $searchProductV5Param) { header { totalData responseCode keywordProcess keywordIntention componentID isQuerySafe additionalParams backendFilters backendFiltersToggle meta { dynamicFields __typename } __typename } data { totalDataText banner { position text url imageURL componentID trackingOption __typename } redirection { url applink __typename } related { relatedKeyword position trackingOption otherRelated { keyword url applink componentID products { oldId: id id: id_str_auto_ name url applink mediaURL { image __typename } shop { oldId: id id: id_str_auto_ name city tier __typename } badge { id title url __typename } price { text number __typename } freeShipping { url __typename } labelGroups { id position title type url styles { key value __typename } __typename } rating wishlist ads { id productClickURL productViewURL productWishlistURL tag __typename } meta { oldWarehouseID: warehouseID warehouseID: warehouseID_str_auto_ componentID oldParentID: parentID parentID: parentID_str_auto_ __typename } __typename } __typename } __typename } suggestion { currentKeyword suggestion query text componentID trackingOption __typename } shopWidget { headline { badge { id title url __typename } shop { id ttsSellerID location City name ratingScore imageShop { sURL __typename } products { id id_str_auto_ ttsProductID name url rating mediaURL { image image300 videoCustom __typename } shop { oldId: id id: id_str_auto_ ttsSellerID name city __typename } price { text number range discountPercentage original __typename } labelGroups { id position title type url styles { key value __typename } __typename } meta { oldParentID: parentID parentID: parentID_str_auto_ isPortrait oldWarehouseID: warehouseID warehouseID: warehouseID_str_auto_ __typename } stock { ttsSKUID __typename } __typename } __typename } __typename } meta { redirect __typename } __typename } ticker { id text query applink componentID trackingOption __typename } violation { headerText descriptionText imageURL ctaURL ctaApplink buttonText buttonType __typename } products { oldId: id id: id_str_auto_ ttsProductID name url applink mediaURL { image image300 videoCustom __typename } shop { oldId: id id: id_str_auto_ ttsSellerID name url city tier __typename } stock { ttsSKUID __typename } badge { id title url __typename } price { text number range original discountPercentage __typename } freeShipping { url __typename } labelGroups { id position title type url styles { key value __typename } __typename } labelGroupsVariant { title type typeVariant hexColor __typename } category { oldId: id id: id_str_auto_ name breadcrumb gaKey __typename } rating wishlist ads { id productClickURL productViewURL productWishlistURL tag __typename } meta { oldParentID: parentID parentID: parentID_str_auto_ oldWarehouseID: warehouseID warehouseID: warehouseID_str_auto_ isImageBlurred isPortrait __typename } __typename } __typename } __typename } }"
Q_REVIEW = "query productReviewList($productURL: String!, $page: Int!, $limit: Int!, $sortBy: String, $filterBy: String, $opt: String) { productrevGetProductReviewList( productID: \"\" productURL: $productURL page: $page limit: $limit sortBy: $sortBy filterBy: $filterBy opt: $opt ) { productID list { id: feedbackID variantName message productRating reviewCreateTime reviewCreateTimestamp isReportable isAnonymous videoAttachments { attachmentID videoUrl __typename } imageAttachments { attachmentID imageThumbnailUrl imageUrl __typename } reviewResponse { message createTime __typename } user { userID fullName image url label __typename } likeDislike { totalLike likeStatus __typename } stats { key formatted count __typename } badRatingReasonFmt __typename } shop { shopID name url image __typename } variantFilter { isUnavailable ticker __typename } hasNext __typename } }"

## Fungsi Ekstraksi GQL
Fungsi ini memanggil API Tokopedia langsung dan mengurai JSON-nya.

In [ ]:
def fetch_search(keyword, sort_option=23):
    # MENGGUNAKAN SORT OPTIONS (ob) MENGGANTIKAN PAGE (page)
    param_str = f"device=mobile&enable_lite_deduplication=true&enter_method=normal_search&l_name=sre&navsource=&ob={sort_option}&page=1&q={keyword.replace(' ', '+')}&rows=60&source=search&srp_component_id=02.01.00.00&srp_page_id=&srp_page_title=&unique_id=93ca95a7632139e8de495d9aad46a929&use_page=true&user_addressId=&user_cityId=176&user_districtId=2274&user_id=12787827&user_lat=0&user_long=0&user_postCode=&user_warehouseId=0&warehouses="
    payload = [{
        "operationName": "SearchProductV5Query",
        "variables": {"cursor":"", "searchProductV5Param": param_str},
        "query": Q_SEARCH
    }]
    
    try:
        r = session.post("https://gql.tokopedia.com/graphql/SearchProductV5Query", headers=HEADERS, json=payload, timeout=20)
        data = r.json()
        if isinstance(data, list): data = data[0]
        if "errors" not in data:
            return data.get("data",{}).get("searchProductV5",{}).get("data",{}).get("products",[])
        else:
            print("Search API Error:", data["errors"][0]["message"])
    except Exception as e:
        print("HTTP Error:", e)
    return []

def fetch_reviews(product_url, limit=40):
    clean_url = product_url.split("?")[0]
    
    payload = [{
        "operationName": "productReviewList",
        "variables": {
            "productURL": clean_url,
            "page": 1, "limit": limit, "sortBy": "informative_score desc", "filterBy": "", "opt": ""
        },
        "query": Q_REVIEW
    }]
    try:
        r = session.post("https://gql.tokopedia.com/graphql/productReviewList", headers=HEADERS, json=payload, timeout=20)
        data = r.json()
        if isinstance(data, list): data = data[0]
        if "errors" not in data:
            return data.get("data",{}).get("productrevGetProductReviewList",{})
    except Exception:
        pass
    return {}

In [ ]:
all_records = []
seen_products = set() 

for keyword in KEYWORDS:
    print(f"\n=== Ekstraksi Keyword: {keyword} ===")
    
    keyword_reviews_count = 0
    
    for sort_opt in SORT_OPTIONS:
        if keyword_reviews_count >= TARGET_PER_KEYWORD:
            print(f"[✓] Target {TARGET_PER_KEYWORD} baris untuk '{keyword}' sudah tercapai!")
            break
            
        products = fetch_search(keyword, sort_option=sort_opt)
        print(f"Mode Sort {sort_opt}: Ditemukan {len(products)} produk.")
        
        for idx, prod in enumerate(products):
            if keyword_reviews_count >= TARGET_PER_KEYWORD:
                break
                
            p_name = prod.get("name", "")
            p_url = prod.get("url", "")
            
            # Cegah duplikasi jika produk ini sudah pernah kita scrape di mode Sort sebelumnya
            if p_url in seen_products:
                continue
            seen_products.add(p_url)
            
            p_price = str(prod.get("price", {}).get("number", 0))
            p_rating = str(prod.get("rating", "0"))
            
            # Cari Lokasi dari toko. 
            shop_city = prod.get("shop", {}).get("city", "")
            p_loc = shop_city if shop_city else "Unknown"
            if p_loc == "Unknown":
                shop_name = prod.get("shop", {}).get("name", "").lower()
                if "tangerang" in shop_name: p_loc = "Tangerang"
                elif "jakarta" in shop_name: p_loc = "Jakarta"
                elif "surabaya" in shop_name: p_loc = "Surabaya"
                elif "bandung" in shop_name: p_loc = "Bandung"
                elif "official" in shop_name: p_loc = "Dilayani Tokopedia"
            
            num_sold = "0"
            for label in prod.get("labelGroups", []):
                if label.get("position") == "ri_product_credibility":
                    num_sold_raw = str(label.get("title", ""))
                    num_sold = __import__("re").sub(r'[^0-9]', '', num_sold_raw)
                    if not num_sold: num_sold = "0"
            
            print(f"  [{idx+1}/{len(products)}] Ekstrak Ulasan: {p_name[:30]}... ({keyword_reviews_count}/{TARGET_PER_KEYWORD})")
            
            # Anti-Bot Jeda
            time.sleep(random.uniform(1.0, 2.0))
            
            # Ambil ulasan menggunakan URL bersih
            rev_data = fetch_reviews(p_url, limit=REVIEWS_PER_PRODUCT)
            rev_list = rev_data.get("list", [])
            total_review = str(len(rev_list)) if rev_list else "Unknown"
            
            for r in rev_list:
                if keyword_reviews_count >= TARGET_PER_KEYWORD:
                    break
                        
                c_review = (r.get("message") or "").strip()
                c_review = c_review.replace("\n", " ").replace("\r", " ")
                    
                if c_review:
                    # Menghitung jumlah kata dengan membuang tanda baca terlebih dahulu
                    clean_text = __import__("re").sub(r"[^a-zA-Z]+", " ", c_review).strip()
                    word_count = len(clean_text.split())
                    
                    if word_count >= 15:
                        c_rating = str(r.get("productRating", "0"))
                        all_records.append({
                            "Category": keyword, "Product Name": p_name, "Location": p_loc, "Price": p_price,
                            "Overall Rating": p_rating, "Number Sold": num_sold, "Total Review": total_review,
                            "Customer Rating": c_rating, "Customer Review": c_review
                        })
                        keyword_reviews_count += 1
                        
print("\nSelesai mengekstrak data! Total Semua Baris:", len(all_records))


=== Ekstraksi Keyword: Minyak Goreng 2 Liter ===
Mode Sort 23: Ditemukan 13 produk.
  [1/13] Ekstrak Ulasan: SunCo Minyak Goreng 2L x 6 PCS... (0/392)
  [2/13] Ekstrak Ulasan: SunCo Minyak Goreng 2 Liter - ... (0/392)
  [3/13] Ekstrak Ulasan: SunCO Minyak Goreng 2 Liter Ke... (4/392)
  [4/13] Ekstrak Ulasan: SunCo Minyak Goreng 2L - Karto... (4/392)
  [5/13] Ekstrak Ulasan: Promo SUNCO Minyak Goreng 2 Lt... (4/392)
  [6/13] Ekstrak Ulasan: SunCo Minyak Goreng 2L Bening ... (4/392)
  [7/13] Ekstrak Ulasan: SUNCO COOKING OIL REFILL 2 LIT... (4/392)
  [8/13] Ekstrak Ulasan: Sunco Minyak Goreng Pouch 2 L... (11/392)
  [9/13] Ekstrak Ulasan: minyak goreng rosebrand 2 lite... (11/392)
  [10/13] Ekstrak Ulasan: minyak sunco 2liter 1karton is... (11/392)
  [11/13] Ekstrak Ulasan: Sunco Minyak Goreng 2 Liter 1 ... (11/392)
  [12/13] Ekstrak Ulasan: MINYAK GORENG SANIA COOKING OI... (11/392)
  [13/13] Ekstrak Ulasan: SOVIA COOKING OIL REFILL 2 LIT... (16/392)
Mode Sort 9: Ditemukan 15 produk.
 

In [8]:
import pandas as pd

OUTPUT_FILE_FIX = "dataset/raw-2_tokopedia_dataset.csv"

if all_records:
    df = pd.DataFrame(all_records)
    
    # Atur kolom sesuai dataset standar
    columns_order = ['Category', 'Product Name', 'Location', 'Price', 'Overall Rating', 'Number Sold', 'Total Review', 'Customer Rating', 'Customer Review']
    df = df.reindex(columns=columns_order)
    
    df.to_csv(OUTPUT_FILE_FIX, index=False, encoding='utf-8-sig')
    print(f"\nBerhasil menyimpan {len(df)} baris ulasan ke {OUTPUT_FILE_FIX}!")
    display(df.head())
else:
    print("Variabel all_records kosong.")



Berhasil menyimpan 19934 baris ulasan ke dataset/raw-2_tokopedia_dataset.csv!


,Category,Product Name,Location,Price,Overall Rating,Number Sold,Total Review,Customer Rating,Customer Review
0,Minyak Goreng 2 Liter,SunCo Minyak Goreng 2 Liter - Minyak Goreng Ba...,Surabaya,59500,5.0,1,34,5,"alhamdulillah udah sampek ,bneran 2liter lhoo,..."
1,Minyak Goreng 2 Liter,SunCo Minyak Goreng 2 Liter - Minyak Goreng Ba...,Surabaya,59500,5.0,1,34,5,Paket minyak sunco di terima dgn baik... Minya...
2,Minyak Goreng 2 Liter,SunCo Minyak Goreng 2 Liter - Minyak Goreng Ba...,Surabaya,59500,5.0,1,34,5,Mantap dapat harga murce ditengah2 mahalnya mi...
3,Minyak Goreng 2 Liter,SunCo Minyak Goreng 2 Liter - Minyak Goreng Ba...,Surabaya,59500,5.0,1,34,5,Kualitas produk sangat baik serta original Har...
4,Minyak Goreng 2 Liter,SUNCO COOKING OIL REFILL 2 LITER - MINYAK GORENG,Unknown,49210,5.0,3,30,5,"barangnya sesuai dengan pesanan, pengiriman ba..."
